# BaseLineGAT 训练与可视化（可复现）

这个 notebook 目标：
- 构图（TF + PPI + STRING，可选 directed）
- 加载 RFA 数据（默认 Landmark 978）
- 训练 BaseLineGAT
- 输出 learning curves（loss/MSE/PCC）与 train/test 指标


In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

ROOT = "/Users/liuxi/Desktop/RFA_GNN"
if not os.path.exists(ROOT):
    ROOT = os.getcwd()
SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import importlib
import data_loader
importlib.reload(data_loader)

from base_gnn import BaseLineGAT
from data_loader import load_rfa_data, build_combined_gnn
from train_deepcop import cold_split

print("ROOT=", ROOT)
print("data_loader=", data_loader.__file__)
print("tf=", tf.__version__)


ROOT= /Users/liuxi/Desktop/RFA_GNN
data_loader= /Users/liuxi/Desktop/RFA_GNN/src/data_loader.py
tf= 2.20.0


In [2]:
debug = True
max_samples = 2000 if debug else 20000
epochs = 2 if debug else 20
batch_size = 32

tf_path = os.path.join(ROOT, "data/omnipath/omnipath_tf_regulons.csv")
ppi_path = os.path.join(ROOT, "data/omnipath/omnipath_interactions.csv")
string_path = os.path.join(ROOT, "data/string_interactions_mapped.csv")
full_gene_path = os.path.join(ROOT, "data/GSE92742_Broad_LINCS_gene_info.txt")
siginfo_path = os.path.join(ROOT, "data/siginfo_beta.txt")
landmark_path = os.path.join(ROOT, "data/landmark_genes.json")
ctl_path = os.path.join(ROOT, "data/cmap/level3_beta_ctl_n188708x12328.h5")
trt_path = os.path.join(ROOT, "data/cmap/level3_beta_trt_cp_n1805898x12328.h5")

for p in [tf_path, ppi_path, string_path, full_gene_path, siginfo_path, landmark_path, ctl_path, trt_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(p)


In [3]:
with open(landmark_path, "r") as f:
    lm = json.load(f)
if isinstance(lm, dict) and "landmark_genes" in lm:
    lm_list = lm["landmark_genes"]
elif isinstance(lm, list):
    lm_list = lm
else:
    lm_list = []

landmark_genes = [str(g.get("entrez_id") or g.get("pr_gene_id")) for g in lm_list if isinstance(g, dict) and (g.get("entrez_id") or g.get("pr_gene_id"))]
landmark_genes = [g for g in landmark_genes if g and g != "None"]
print("landmark genes:", len(landmark_genes))


landmark genes: 978


In [4]:
adj_matrix, node_list, gene2idx, edge_index = build_combined_gnn(
    tf_path=tf_path,
    ppi_path=ppi_path,
    string_path=string_path,
    full_gene_path=full_gene_path,
    landmark_path=landmark_path,
    landmark_genes=landmark_genes,
    confid_threshold=0.9,
    directed=False,
)
print("graph nodes:", len(node_list), "edges:", int(edge_index.shape[1]))
print("adj nonzero:", int((adj_matrix != 0).sum()))


>>> 正在构建 Combined GNN (TF + PPI) ...
正在加载全基因映射: /Users/liuxi/Desktop/RFA_GNN/data/GSE92742_Broad_LINCS_gene_info.txt
加载 TF 调控网络: /Users/liuxi/Desktop/RFA_GNN/data/omnipath/omnipath_tf_regulons.csv
TF 边数: 1276
加载 PPI 网络: /Users/liuxi/Desktop/RFA_GNN/data/omnipath/omnipath_interactions.csv
PPI 边数: 1605
加载 STRING 网络: /Users/liuxi/Desktop/RFA_GNN/data/string_interactions_mapped.csv
STRING 添加的边数: 1930
  STRING 边数: 1930
Combined Graph 构建完成: 978 节点, 3270 边 (含权重)
graph nodes: 978 edges: 5285
adj nonzero: 6125


In [5]:
data = load_rfa_data(
    ctl_path,
    trt_path,
    landmark_path=landmark_path,
    siginfo_path=siginfo_path,
    use_landmark_genes=True,
    full_gene_path=full_gene_path,
    max_samples=max_samples,
)
if data is None:
    raise RuntimeError("load_rfa_data returned None")

print("X_ctl:", data["X_ctl"].shape)
print("y_delta:", data["y_delta"].shape)
print("X_drug:", data["X_drug"].shape)
print("loss_mask:", np.asarray(data["loss_mask"]).shape)


正在加载 RFA 数据 (Landmark Mode: True)...
CSV路径: CTL=/Users/liuxi/Desktop/RFA_GNN/data/cmap/level3_beta_ctl_n188708x12328.h5, TRT=/Users/liuxi/Desktop/RFA_GNN/data/cmap/level3_beta_trt_cp_n1805898x12328.h5
正在加载全基因映射: /Users/liuxi/Desktop/RFA_GNN/data/GSE92742_Broad_LINCS_gene_info.txt
目标基因数: 978
正在读取元数据: /Users/liuxi/Desktop/RFA_GNN/data/siginfo_beta.txt
过滤后元数据记录数: 117284
已构建 Control 查找表，覆盖 401 个 (Cell, Time, Batch) 组合。
>>> 进入 DeepCOP 对齐模式: 计算 Cell-Specific Mean Control
发现 57929 个 Control 样本，涉及 162 个细胞系。
正在加载所有 Control 数据以计算均值...
正在加载数据文件: /Users/liuxi/Desktop/RFA_GNN/data/cmap/level3_beta_ctl_n188708x12328.h5 ...
  H5 (h5py): 匹配到 57929 个样本 (axis0/Index). 读取中...
  读取耗时: 22.04s
  Gene ID 匹配: Index=0, Columns=978
  形状检查: Rows=57929, Target=978
  已筛选并对齐到 978 个 Landmark Genes。
已计算 162 个细胞系的基准表达谱。
发现 67663 个高质量 Treatment 样本。
超过内存限制 (2000)，随机采样中...
正在加载 Treatment 数据...
正在加载数据文件: /Users/liuxi/Desktop/RFA_GNN/data/cmap/level3_beta_trt_cp_n1805898x12328.h5 ...
  H5 (h5py): 匹配到 2000 个样本 (axis0/Index)

KeyboardInterrupt: 

In [ ]:
train_data, test_data = cold_split(data)
train_ctl, train_trt, train_drug = train_data
test_ctl, test_trt, test_drug = test_data

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
cell_idx = le.fit_transform(data["cell_names"])
num_cells = int(len(le.classes_))

unique_drugs = np.unique(data["drug_ids"])
np.random.seed(42)
test_drugs = np.random.choice(unique_drugs, int(len(unique_drugs) * 0.2), replace=False)
test_mask = np.isin(data["drug_ids"], test_drugs)
train_mask = ~test_mask

train_cells = cell_idx[train_mask]
test_cells = cell_idx[test_mask]

print("train samples:", train_ctl.shape[0], "test samples:", test_ctl.shape[0], "num_cells:", num_cells)


In [ ]:
num_genes = int(adj_matrix.shape[0])
model = BaseLineGAT(
    num_genes=num_genes,
    num_cells=num_cells,
    hidden_dim=64,
    num_heads=4,
    dropout=0.2,
    use_residual=False,
    use_drug_embedding=False,
)

loss_mask = tf.constant(data["loss_mask"], dtype=tf.float32)

def pcc_loss(y_true, y_pred):
    mx = tf.reduce_mean(y_true, axis=1, keepdims=True)
    my = tf.reduce_mean(y_pred, axis=1, keepdims=True)
    xm = y_true - mx
    ym = y_pred - my
    r_num = tf.reduce_sum(xm * ym, axis=1)
    r_den = tf.sqrt(tf.reduce_sum(tf.square(xm), axis=1) * tf.reduce_sum(tf.square(ym), axis=1) + 1e-8)
    r = r_num / r_den
    return 1.0 - tf.reduce_mean(r)

def masked_combined_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    mask = tf.cast(loss_mask, tf.float32)
    mse = tf.reduce_sum(tf.square(y_true - y_pred) * mask)
    valid_count = tf.reduce_sum(mask)
    mse = mse / tf.maximum(valid_count, 1.0)
    valid_indices = tf.where(loss_mask[0] > 0)[:, 0]
    yt = tf.gather(y_true, valid_indices, axis=1)
    yp = tf.gather(y_pred, valid_indices, axis=1)
    pcc = pcc_loss(yt, yp)
    return mse + 5.0 * pcc

class GATWrapper(keras.Model):
    def __init__(self, gat_model, adj_matrix):
        super().__init__()
        self.gat = gat_model
        self.adj = tf.constant(adj_matrix, dtype=tf.float32)

    def call(self, inputs):
        ctl, drug_targets, cell_idx = inputs
        cell_idx = tf.cast(cell_idx, tf.int32)
        return self.gat([self.adj, ctl, drug_targets, cell_idx])

wrapped_model = GATWrapper(model, adj_matrix)
wrapped_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss=masked_combined_loss,
    metrics=[keras.metrics.MeanSquaredError()],
)

print("model compiled")


In [ ]:
def eval_pcc_mse(model, ctl, drug, cells, y_true, loss_mask, batch_size=32, max_eval=None):
    if max_eval is not None and len(ctl) > int(max_eval):
        rng = np.random.default_rng(0)
        idx = rng.choice(len(ctl), size=int(max_eval), replace=False)
        ctl = ctl[idx]
        drug = drug[idx]
        cells = cells[idx]
        y_true = y_true[idx]
    pred = model.predict([ctl, drug, cells], batch_size=batch_size, verbose=0)
    valid_indices = np.where(loss_mask[0] > 0)[0]
    y_true_valid = y_true[:, valid_indices]
    pred_valid = pred[:, valid_indices]
    pcc_list = []
    for i in range(len(y_true_valid)):
        yt = y_true_valid[i]
        yp = pred_valid[i]
        if np.std(yt) > 1e-6 and np.std(yp) > 1e-6:
            p, _ = pearsonr(yt, yp)
            pcc_list.append(p)
    avg_pcc = float(np.mean(pcc_list)) if pcc_list else 0.0
    mse = float(mean_squared_error(y_true_valid, pred_valid))
    return {"mse": mse, "pcc": avg_pcc}

class PCCCallback(keras.callbacks.Callback):
    def __init__(self, loss_mask, train_data, val_data, batch_size=32, max_eval=2048):
        super().__init__()
        self.loss_mask = loss_mask
        self.train_data = train_data
        self.val_data = val_data
        self.batch_size = int(batch_size)
        self.max_eval = int(max_eval) if max_eval is not None else None
        self.train_pcc = []
        self.val_pcc = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        ctl_tr, drug_tr, cell_tr, y_tr = self.train_data
        ctl_va, drug_va, cell_va, y_va = self.val_data
        tr = eval_pcc_mse(self.model, ctl_tr, drug_tr, cell_tr, y_tr, self.loss_mask, batch_size=self.batch_size, max_eval=self.max_eval)
        va = eval_pcc_mse(self.model, ctl_va, drug_va, cell_va, y_va, self.loss_mask, batch_size=self.batch_size, max_eval=self.max_eval)
        self.train_pcc.append(tr["pcc"])
        self.val_pcc.append(va["pcc"])
        logs["pcc"] = tr["pcc"]
        logs["val_pcc"] = va["pcc"]
        print(f"Epoch {epoch+1}: pcc={tr['pcc']:.4f} val_pcc={va['pcc']:.4f}")

pcc_cb = PCCCallback(
    loss_mask=data["loss_mask"],
    train_data=(train_ctl, train_drug, train_cells, train_trt),
    val_data=(test_ctl, test_drug, test_cells, test_trt),
    batch_size=batch_size,
    max_eval=2048,
)


In [ ]:
history = wrapped_model.fit(
    [train_ctl, train_drug, train_cells],
    train_trt,
    epochs=int(epochs),
    batch_size=int(batch_size),
    callbacks=[pcc_cb],
    validation_data=([test_ctl, test_drug, test_cells], test_trt),
)


In [ ]:
import matplotlib.pyplot as plt

hist = history.history
ep = np.arange(1, len(hist.get("loss", [])) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(ep, hist.get("loss", []), label="train_loss")
if "val_loss" in hist:
    axes[0].plot(ep, hist.get("val_loss", []), label="val_loss")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(ep, hist.get("mean_squared_error", []), label="train_mse")
if "val_mean_squared_error" in hist:
    axes[1].plot(ep, hist.get("val_mean_squared_error", []), label="val_mse")
axes[1].set_title("MSE")
axes[1].legend()

ep_pcc = np.arange(1, len(pcc_cb.train_pcc) + 1)
axes[2].plot(ep_pcc, pcc_cb.train_pcc, label="train_pcc")
axes[2].plot(ep_pcc, pcc_cb.val_pcc, label="val_pcc")
axes[2].set_title("Sample-wise PCC (subset)")
axes[2].legend()

plt.show()


In [ ]:
train_metrics = eval_pcc_mse(wrapped_model, train_ctl, train_drug, train_cells, train_trt, data["loss_mask"], batch_size=batch_size, max_eval=20000)
test_metrics = eval_pcc_mse(wrapped_model, test_ctl, test_drug, test_cells, test_trt, data["loss_mask"], batch_size=batch_size, max_eval=None)
print(f"Train | MSE: {train_metrics['mse']:.4f} | Sample-wise PCC: {train_metrics['pcc']:.4f}")
print(f"Test  | MSE: {test_metrics['mse']:.4f} | Sample-wise PCC: {test_metrics['pcc']:.4f}")

pred = wrapped_model.predict([test_ctl, test_drug, test_cells], batch_size=batch_size, verbose=0)
valid_indices = np.where(np.asarray(data["loss_mask"])[0] > 0)[0]
test_trt_valid = test_trt[:, valid_indices]
pred_valid = pred[:, valid_indices]
flat_pcc = pearsonr(test_trt_valid.flatten(), pred_valid.flatten())[0]
print(f"Test Global PCC (flatten): {float(flat_pcc):.4f}")
